# Boss Agent Demo

The boss agent is architecturally different from a specialist: it has no domain tools of its
own and never touches Delta tables directly. Its job is orchestration and synthesis, built as
a **LangGraph** graph with three nodes:

```
select_specialists  ->  run_specialists  ->  synthesize
```

1. **select_specialists** — an LLM call (Groq, `llama-3.3-70b-versatile`) reads the query and
   the roster of *actually registered* specialists (`backend/agents/boss/registry.py`) and
   decides which ones are relevant. It can select none — it isn't forced to route every query
   to Finance just because Finance is the only specialist that exists yet.
2. **run_specialists** — calls `.run(query)` on each selected specialist via a thread pool.
   Only one specialist exists today, but this is structured for N specialists running in
   parallel once the rest are built.
3. **synthesize** — a second LLM call takes every specialist's `Finding`s and writes a board
   memo, explicitly instructed to flag disagreement between agents rather than smoothing it
   into false consensus, and to cite the agent/tool behind every claim.

**Why Groq, not a paid API**: the boss agent runs an LLM call every session (not per tool
call), so cost matters far less than it would per-specialist — but for a portfolio project
still in active development, a free tier avoids burning money on every test run. Groq's free
tier serves capable open models (Llama 3.3 70B) fast enough for orchestration and synthesis.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.boss import BossAgent
from backend.db import DatabricksClient

db = DatabricksClient()
boss = BossAgent(db)

## Query 1 — relevant to Finance

Expect `select_specialists` to route this to the Finance agent.

In [2]:
rec = boss.run("How is our financial health looking?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Requires human approval: {rec.requires_human_approval}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.action_items:
    print("\nAction items:")
    for item in rec.action_items:
        print(f"  - {item}")

Agents invoked: ['Finance']
Requires human approval: True
Overall confidence: 0.75

Synthesis:
**Board Memo – Financial Health Assessment**

1. **Contribution Margin** – Finance’s `calculate_margin_trend` indicates an improving contribution margin over the last 12 months (confidence 0.75). This suggests that unit economics are strengthening.

2. **Revenue Anomalies** – The `detect_revenue_anomalies` tool flagged 13 anomalous daily revenue points out of 614 days (confidence 0.85). These anomalies could indicate data integrity issues or irregular business events that warrant further investigation.

3. **Order Failure & Refund Risk** – The `payment_failure_rate` tool reports a 1.24% failure rate based on a proxy of canceled/unavailable orders (confidence 0.60). Additionally, the `refund_impact_analysis` tool shows that 1.68% of total payment value ($269,735.11) is at risk from these cancellations (confidence 0.60). Both metrics point to a non‑negligible operational risk that could erode r

## Query 2 — irrelevant to any registered specialist

This tests that agent selection is a real decision, not a rubber stamp. With only Finance registered, an HR-flavored question should route to *no* specialist rather than forcing Finance to answer something outside its domain.

In [3]:
rec2 = boss.run("What's the current employee satisfaction across departments?")

print(f"Agents invoked: {[a.value for a in rec2.agents_invoked]}")
print(f"\nSynthesis:\n{rec2.synthesis}")

Agents invoked: []

Synthesis:
No specialist agent currently available is relevant to this query.


## What's pending

- **Dissent detection** (`rec.dissents`) can't produce anything meaningful yet — it needs 2+ specialists reporting on the same query for there to be anything to disagree about. Worth re-running this notebook once Sales or Risk exist alongside Finance.
- **`GovernanceLog` persistence** — `BoardRecommendation` is fully validated here but not yet written to MongoDB for audit trail (planned, step 7 of the build order).
- **`request_clarification()`** — the boss agent's only planned domain tool (loop back to a specialist for more detail on an ambiguous query) isn't built yet.